# Script 4_5 — Laboratory files and the biomarker panel

Merges the NHANES laboratory files with the questionnaire master file from
script 4, computes the systemic immune-inflammation index, and assembles the
ten-marker panel from which script 5_5 computes Klemera–Doubal biological
age.

## Inputs

- `data/raw/2011-2012/` and `data/raw/2013-2014/` (file names matched
  without regard to case): `CBC`, `BIOPRO`, `GHB`, `TCHOL`, and the optional
  `GLU`, `FASTQX`, `ALB_CR`. The optional three are carried into the master
  file but enter no analysis; the notebook skips them if absent.
- `data/processed/5_paday_dp_sl_cf_paq.csv` — script 4.

## Output

- `data/processed/5_paday_dp_sl_cf_paq_bio.csv` — one row per participant,
  questionnaires and biomarkers.

## Derived variables

Systemic immune-inflammation index from the complete blood count (Hu et al.,
2014): SII = platelets × neutrophils / lymphocytes (×10⁹/L), with the
neutrophil-to-lymphocyte and monocyte-to-lymphocyte ratios alongside, and
the natural log of each (`SII_log`, `NLR_log`, `MLR_log`). Only SII and its
log enter the models; NLR is described in Supplementary Table S13.

Biomarker panel for KDM biological age (script 5_5): albumin, creatinine,
alkaline phosphatase, glucose, glycated haemoglobin, total cholesterol,
lymphocyte percentage, mean corpuscular volume, red cell distribution width
and white cell count. C-reactive protein was not measured in NHANES
2011–2014, so it is not in the panel.

## References

Hu B, Yang X-R, Xu Y, et al. (2014) Systemic immune-inflammation index
predicts prognosis of patients after curative resection for hepatocellular
carcinoma. *Clinical Cancer Research* 20:6212–6222.

Klemera P, Doubal S (2006) A new approach to the concept and computation of
biological age. *Mechanisms of Ageing and Development* 127:240–248.

## Step 0 — Imports and paths


In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 50
pd.options.display.width = 140


def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')


BASE = find_root()
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'
sys.path.insert(0, str(BASE / 'scripts'))
from config import raw_file

MASTER_IN  = PROC / '5_paday_dp_sl_cf_paq.csv'
MASTER_OUT = PROC / '5_paday_dp_sl_cf_paq_bio.csv'
print(f'project root: {BASE}')
print(f'input : {MASTER_IN.name}')
print(f'output: {MASTER_OUT.name}')

## Step 1 — Load the laboratory files

Each cycle contributes the same files (suffix `_G` for 2011–2012, `_H` for
2013–2014); the two cycles are stacked row-wise. An absent optional file is
skipped.

In [ ]:
LAB_FILES      = ['CBC', 'BIOPRO', 'GHB', 'TCHOL']       # used by the analyses
OPTIONAL_FILES = ['GLU', 'FASTQX', 'ALB_CR']              # carried, not analysed

def load_pair(name):
    g = pd.read_sas(raw_file(RAW / '2011-2012', f'{name}_G.xpt'), format='xport')
    h = pd.read_sas(raw_file(RAW / '2013-2014', f'{name}_H.xpt'), format='xport')
    g['CYCLE'] = '2011-2012'
    h['CYCLE'] = '2013-2014'
    df = pd.concat([g, h], ignore_index=True)
    df['SEQN'] = df['SEQN'].astype(int)
    return df

labs = {}
for name in LAB_FILES + OPTIONAL_FILES:
    try:
        labs[name] = load_pair(name)
        print(f'  {name:8s}  n = {labs[name].shape[0]:5d}  cols = {labs[name].shape[1]:2d}')
    except FileNotFoundError as e:
        if name in LAB_FILES:
            raise
        print(f'  {name:8s}  skipped ({e})')

## Step 2 — Select the columns

Only the variables needed for the immune-inflammation index and the KDM
biological-age panel are kept, plus the optional carried variables.

| file | variables retained |
|---|---|
| CBC | white cell count, lymphocyte percentage, MCV, RDW, lymphocyte, neutrophil, monocyte and platelet counts |
| BIOPRO | albumin, creatinine, alkaline phosphatase, serum glucose, globulin |
| GHB | glycated haemoglobin |
| TCHOL | total cholesterol |
| GLU (optional) | fasting glucose, fasting duration, fasting-subsample weight |
| FASTQX (optional) | fasting duration |
| ALB_CR (optional) | urinary albumin-to-creatinine ratio |

In [ ]:
KEEP_COLS = {
    'CBC':    ['SEQN', 'LBXWBCSI', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW',
               'LBDLYMNO', 'LBDNENO', 'LBDMONO', 'LBXPLTSI'],
    'BIOPRO': ['SEQN', 'LBXSAL', 'LBXSCR', 'LBXSAPSI', 'LBXSGL', 'LBXSGB'],
    'GHB':    ['SEQN', 'LBXGH'],
    'TCHOL':  ['SEQN', 'LBXTC'],
    'GLU':    ['SEQN', 'LBXGLU', 'PHAFSTHR', 'WTSAF2YR'],
    'FASTQX': ['SEQN', 'PHAFSTHR'],
    'ALB_CR': ['SEQN', 'URDACT'],
}

subs = {}
for name, cols in KEEP_COLS.items():
    if name not in labs:
        continue
    df = labs[name]
    keep = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f'  {name}: {missing} not found, dropped')
    subs[name] = df[keep].copy()
    print(f'  {name:8s}  retained {len(keep):2d} cols, n = {subs[name].shape[0]:5d}')

# Both FASTQX and GLU carry PHAFSTHR; keep the version from GLU (fasting subsample)
if 'FASTQX' in subs and 'GLU' in subs:
    subs['FASTQX'] = subs['FASTQX'][[c for c in subs['FASTQX'].columns if c != 'PHAFSTHR']]


## Step 3 — Immune-inflammation index

SII, NLR and MLR from the absolute cell counts, with natural-log versions.

In [ ]:
cbc = subs['CBC'].copy()
# Zero lymphocyte counts -> NaN (undefined ratios)
lym = cbc['LBDLYMNO'].replace(0, np.nan)
cbc['SII'] = cbc['LBXPLTSI'] * cbc['LBDNENO'] / lym
cbc['NLR'] = cbc['LBDNENO']  / lym
cbc['MLR'] = cbc['LBDMONO']  / lym
for col in ['SII', 'NLR', 'MLR']:
    cbc[f'{col}_log'] = np.where(cbc[col] > 0, np.log(cbc[col]), np.nan)
subs['CBC'] = cbc

print('Distribution of derived indices:')
print(cbc[['SII', 'NLR', 'MLR']].describe().round(2).to_string())


## Step 4 — Combine the laboratory subsets

Outer joins on `SEQN` keep every participant with at least one laboratory
measurement.

In [ ]:
bio = subs['CBC']
for name in ['BIOPRO', 'GHB', 'TCHOL', 'GLU', 'FASTQX', 'ALB_CR']:
    if name in subs:
        bio = bio.merge(subs[name], on='SEQN', how='outer')

print(f'Biomarker frame: {bio.shape}')
print(f'Unique SEQN    : {bio["SEQN"].nunique()}')


## Step 5 — Merge into the participant master

Left join on `SEQN`: every participant in the questionnaire master is kept;
those without laboratory data receive missing values in the biomarker
columns.

In [ ]:
master = pd.read_csv(MASTER_IN)
master['SEQN'] = master['SEQN'].astype(int)
print(f'Master (questionnaires): {master.shape}')

merged = master.merge(bio, on='SEQN', how='left')
print(f'Merged (questionnaires + biomarkers): {merged.shape}')
added = [c for c in merged.columns if c not in master.columns]
print(f'Added columns ({len(added)}): {added}')

merged.to_csv(MASTER_OUT, index=False)
print()
print(f'Wrote {MASTER_OUT}')
print(f'  shape : {merged.shape}')
print(f'  size  : {MASTER_OUT.stat().st_size / 1e6:.1f} MB')


## Step 6 — Missingness

Per-variable missingness in the full sample and among adults.

In [ ]:
audit_cols = ['LBXWBCSI', 'LBDLYMNO', 'LBDNENO', 'LBDMONO', 'LBXPLTSI',
              'LBXLYPCT', 'LBXMCVSI', 'LBXRDW',
              'LBXSAL', 'LBXSCR', 'LBXSAPSI', 'LBXSGL', 'LBXSGB',
              'LBXGH', 'LBXTC', 'LBXGLU', 'PHAFSTHR',
              'SII', 'NLR', 'MLR']

def audit(frame, label):
    rows = [{'variable': c,
             'n_observed': int(frame[c].notna().sum()) if c in frame else 0,
             'missing_pct': round(frame[c].isna().mean() * 100, 1) if c in frame else 100.0}
            for c in audit_cols]
    return pd.DataFrame(rows)

print('Full sample missingness:')
print(audit(merged, 'all').to_string(index=False))

adults = merged[merged['RIDAGEYR'] >= 18].copy()
print(f'\nAdult subsample (RIDAGEYR >= 18): n = {len(adults)}')
print(audit(adults, 'adults').to_string(index=False))


## Step 7 — Integrity checks

In [ ]:
assert merged['SEQN'].is_unique, 'Duplicate SEQN in merged frame'
for col in ['SII', 'NLR', 'MLR']:
    n_inf = int(np.isinf(merged[col]).sum())
    n_nan = int(merged[col].isna().sum())
    print(f'  {col:5s}  inf = {n_inf}  nan = {n_nan}  '
          f'range = [{merged[col].min():.2f}, {merged[col].max():.2f}]')

kdm_inputs = ['LBXSAL', 'LBXSCR', 'LBXSGL', 'LBXGH', 'LBXTC',
              'LBXSAPSI', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI']
complete = merged[merged['RIDAGEYR'] >= 18][kdm_inputs].dropna()
print(f'\nAdults with complete KDM-Biological Age biomarker panel: n = {len(complete)}')
